# Prática 10/11 · Qual modelo acha melhor os municípios de abstenção alta? Validação cruzada
**Aprendizagem Computacional · 4º TIA · Aula 03 · versão INICIAL**

Na prática de 03/11, a regressão logística ganhou do baseline num teste de **36 municípios**. Mas 36 é pouco: um acerto a mais ou a menos muda o recall em 9 pontos.
Hoje a pergunta é outra: **entre vários modelos, qual escolher, sem gastar o teste?** A resposta é a **validação cruzada** (cap. 9 e 18).

| Parte | O que você faz | Apostila |
|---|---|---|
| 1–2 | Carregar, montar X e y e separar (pronto, igual a 03/11) | cap. 9 |
| 3 | Ver o problema: o resultado muda com o sorteio (pronto) | cap. 9 |
| 4 | **TODO 1:** criar os 5 "folds" estratificados | cap. 9 |
| 5 | **TODO 2:** montar k-NN, árvore e Random Forest | cap. 13, 16, 18 |
| 6 | **TODO 3:** comparar todos com `cross_validate` | cap. 9, 18 |
| 7 | **TODO 4:** achar o melhor k do k-NN com `GridSearchCV` | cap. 13 |
| 8 | Pesos de classe (pronto) | cap. 12 |
| 9 | **TODO 5:** escolher e abrir o teste UMA vez | cap. 9 |
| 10 | Verificação final | — |

Não precisa instalar nada no Colab. O notebook não grava nenhum arquivo. `random_state=42` em tudo que sorteia.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEMENTE = 42
LIMIAR_ALTA = 25
FEATURES = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO']
pd.set_option('display.precision', 2)

## 1 · Carregar os dados (pronto)
Os mesmos 142 municípios de MT de 03/11 (TSE, 1º turno de 2026), agora com duas colunas a mais de votos brancos e nulos, que vamos usar em 17/11.
Se a célula falhar no Colab, arraste o CSV da pasta `dados/` para a aba **Arquivos** e rode de novo.

In [ ]:
ARQUIVO = 'municipios_mt_votos.csv'
URL = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/atividades/2026-11-10_Aula03_Validacao_Cruzada/dados/municipios_mt_votos.csv'

candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
local = next((p for p in candidatos if p.exists()), None)
if local is not None:
    municipios = pd.read_csv(local)
    print('Lido do arquivo:', local.resolve())
else:
    municipios = pd.read_csv(URL)
    print('Lido do GitHub da disciplina')
print(municipios.shape)

## 2 · X, y e separação (pronto, igual a 03/11)
O teste fica **guardado** até a parte 9. Tudo que fizermos até lá usa só o treino.

In [ ]:
from sklearn.model_selection import train_test_split

y = (municipios['ABSTENCAO_PCT'] >= LIMIAR_ALTA).astype(int)
X = municipios[FEATURES]
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=SEMENTE, stratify=y)
print('treino:', X_treino.shape, '| teste:', X_teste.shape, '| "alta" no treino:', y_treino.sum())

## 3 · O problema: o resultado depende do sorteio (pronto)
Vamos repetir o experimento de 03/11 (logística, recall no teste) mudando **só** a semente do sorteio treino/teste.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score

recalls = []
for semente in range(10):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.25, random_state=semente, stratify=y)
    modelo = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xa, ya)
    recalls.append(recall_score(yb, modelo.predict(Xb)))
    print(f'semente {semente}: recall no teste = {recalls[-1]:.2f}')
print(f'\nmenor {min(recalls):.2f} | maior {max(recalls):.2f}')

O mesmo modelo, com os mesmos dados, muda de recall só por causa do sorteio (compare o menor e o maior). Se escolhermos o modelo por **um** teste, estaremos escolhendo, em parte, pela sorte.

**Validação cruzada (cap. 9):** dividimos o **treino** em 5 partes (*folds*). O modelo treina em 4 e é avaliado na 5ª, e isso se repete 5 vezes, até cada parte ter sido avaliação uma vez. Saem 5 notas: a **média** diz o desempenho, e o **desvio** diz o quanto ele oscila.
É como no inventário florestal: em vez de medir uma única parcela e extrapolar, medimos várias e olhamos a média e a variação entre elas.

## 4 · TODO 1: os 5 folds estratificados (cap. 9)
Crie `cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)`.
O "Stratified" mantém a proporção de municípios "alta" igual em cada fold, como o `stratify=y` fazia no split.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# TODO 1
cv = None

for i, (idx_treino, idx_valid) in enumerate(cv.split(X_treino, y_treino), start=1):
    print(f'fold {i}: treina em {len(idx_treino)}, avalia em {len(idx_valid)} ({y_treino.iloc[idx_valid].sum()} "alta")')

## 5 · TODO 2: os candidatos (cap. 13, 16 e 18)
O baseline e a logística já estão no dicionário. Complete:
- **k-NN:** `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`. O k-NN mede distância, então **precisa** de escala (cap. 6).
- **Árvore de decisão:** `DecisionTreeClassifier(max_depth=3, random_state=SEMENTE)`. Árvore não precisa de escala: ela só compara "maior ou menor que".
- **Random Forest:** `RandomForestClassifier(n_estimators=200, random_state=SEMENTE)`: 200 árvores, cada uma vendo um sorteio diferente dos dados, votando juntas.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

candidatos = {
    'Baseline': DummyClassifier(strategy='most_frequent'),
    'Logística': make_pipeline(StandardScaler(), LogisticRegression()),
    'k-NN (k=5)': None,        # TODO 2a
    'Árvore (prof. 3)': None,  # TODO 2b
    'Random Forest': None,     # TODO 2c
}

## 6 · TODO 3: comparação justa com `cross_validate` (cap. 9 e 18)
Todos os modelos passam pelo **mesmo** protocolo: mesmos folds, mesmas métricas, só o treino.
Dentro do `for`, chame `cross_validate(modelo, X_treino, y_treino, cv=cv, scoring=METRICAS)`. Ele devolve um dicionário com as notas de cada fold em `test_accuracy`, `test_recall` e `test_f1` (aqui, "test" quer dizer o fold de avaliação, não o nosso teste guardado).

**F1** é a média harmônica de precisão e recall: só fica alta se as duas forem boas. Usamos para escolher.

In [ ]:
from sklearn.model_selection import cross_validate

METRICAS = ['accuracy', 'recall', 'f1']
linhas = []
for nome, modelo in candidatos.items():
    # TODO 3: r = cross_validate(...)
    r = None
    linhas.append({'modelo': nome,
                   'acurácia': r['test_accuracy'].mean(),
                   'recall': r['test_recall'].mean(),
                   'recall (desvio)': r['test_recall'].std(),
                   'F1': r['test_f1'].mean()})

tabela_cv = pd.DataFrame(linhas).set_index('modelo').sort_values('F1', ascending=False)
tabela_cv

**Pergunta para discussão:** a árvore teve o recall com o maior desvio entre os folds. O que isso quer dizer para quem vai usar o modelo?

**Resposta esperada:** que o desempenho dela depende muito de quais municípios caem no treino. Uma árvore pequena muda o primeiro corte com poucos dados diferentes, e as previsões mudam junto: ela é **instável** (alta variância, cap. 10 e 16). O Random Forest junta 200 árvores justamente para reduzir essa oscilação (cap. 18).

## 7 · TODO 4: o melhor k do k-NN com `GridSearchCV` (cap. 13)
O k (quantos vizinhos votam) é um **hiperparâmetro**: nós escolhemos, o modelo não aprende. O `GridSearchCV` testa cada valor da lista com a mesma validação cruzada e guarda o melhor.

Complete: `GridSearchCV(pipeline_knn, grade, cv=cv, scoring='f1')` e depois `.fit(X_treino, y_treino)`.
O nome do parâmetro na grade é `etapa__parametro`: `kneighborsclassifier__n_neighbors`.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipeline_knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
grade = {'kneighborsclassifier__n_neighbors': [1, 3, 5, 7, 9, 11, 15]}

# TODO 4: busca = GridSearchCV(...).fit(...)
busca = None

print('melhor k:', busca.best_params_['kneighborsclassifier__n_neighbors'], '| F1 médio na CV:', round(busca.best_score_, 2))
resultados = pd.DataFrame(busca.cv_results_)
plt.plot(grade['kneighborsclassifier__n_neighbors'], resultados['mean_test_score'], marker='o')
plt.xlabel('k (número de vizinhos)'); plt.ylabel('F1 médio na validação cruzada'); plt.title('Escolhendo o k do k-NN'); plt.show()

## 8 · Pesos de classe (pronto, cap. 12)
Só 31% dos municípios são "alta", então errar um "alta" custa pouco para o modelo na hora de aprender. Com `class_weight='balanced'`, cada erro na classe rara passa a pesar mais (aqui, cerca de 2,2 vezes).
Em 03/11 fizemos algo parecido mexendo no limiar **depois** de treinar. Aqui o ajuste acontece **no treino**.

In [ ]:
logistica_pesos = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced'))
r = cross_validate(logistica_pesos, X_treino, y_treino, cv=cv, scoring=METRICAS)
tabela_cv.loc['Logística (pesos)'] = [r['test_accuracy'].mean(), r['test_recall'].mean(),
                                      r['test_recall'].std(), r['test_f1'].mean()]
tabela_cv.sort_values('F1', ascending=False)

## 9 · TODO 5: escolher e abrir o teste UMA vez (cap. 9)
Pela tabela, o candidato com maior F1 na validação cruzada é a **logística com pesos**. Só agora, com a escolha feita, olhamos o teste:
1. ajuste `logistica_pesos` em **todo** o treino (`fit(X_treino, y_treino)`);
2. preveja `X_teste` e guarde em `prev_final`.

Se a nota no teste decepcionar, **não** volte para trocar de modelo: isso transformaria o teste em mais uma validação, e a nota final deixaria de ser honesta.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix

# TODO 5
modelo_final = None
prev_final = None

print(f'TESTE · logística com pesos: acurácia {accuracy_score(y_teste, prev_final):.2f} | '
      f'precisão {precision_score(y_teste, prev_final):.2f} | recall {recall_score(y_teste, prev_final):.2f} | '
      f'F1 {f1_score(y_teste, prev_final):.2f}')
vn, fp, fn, vp = confusion_matrix(y_teste, prev_final).ravel()
print(f'{vp} de {vp + fn} municípios "alta" encontrados, {fp} alarmes falsos')

**Pergunta para discussão:** por que não escolhemos o modelo pelo resultado no teste, já que ele é "o mais real"?

**Resposta esperada:** porque, se o teste ajuda a escolher, ele deixa de ser independente: escolhemos o que deu sorte naqueles 36 municípios, e a nota fica otimista. A validação cruzada escolhe, e o teste só **confirma**, uma vez. No artigo da PPI, relate as duas coisas: a média ± desvio da CV e a nota final no teste.

## 10 · Verificação final
Rode sem alterar. Os valores esperados estão no `LEIAME.md` da prática.

In [ ]:
def conferir(descricao, teste):
    try:
        passou = bool(teste())
    except Exception:          # variável ainda vazia (None) ou célula não rodada
        passou = False
    print('✅' if passou else '❌', descricao)
    return passou

ok = [
    conferir('cv tem 5 folds estratificados', lambda: cv.get_n_splits() == 5 and type(cv).__name__ == 'StratifiedKFold'),
    conferir('5 candidatos montados (nenhum None)', lambda: all(m is not None for m in candidatos.values())),
    conferir('k-NN (k=5): F1 médio 0,56 na CV', lambda: round(tabela_cv.loc['k-NN (k=5)', 'F1'], 2) == 0.56),
    conferir('Random Forest: F1 médio 0,53 na CV', lambda: round(tabela_cv.loc['Random Forest', 'F1'], 2) == 0.53),
    conferir('GridSearch escolheu k = 5', lambda: busca.best_params_['kneighborsclassifier__n_neighbors'] == 5),
    conferir('teste: recall da logística com pesos = 0,73', lambda: round(recall_score(y_teste, prev_final), 2) == 0.73),
]
print(f'\n{sum(ok)} de {len(ok)} verificações OK')

## Checklist (as 6 perguntas da PPI)
1. **y?** abstenção ≥ 25%. 2. **Baseline?** sempre "baixa" (F1 = 0). 3. **Métrica?** F1 para escolher (equilibra precisão e recall) e recall para relatar.
4. **Split?** teste separado no início; escolha feita só com CV no treino. 5. **Vazamento?** escala dentro do Pipeline, então ela é refeita em cada fold.
6. **Evidência?** relatamos média **e desvio**: com 106 municípios de treino, diferenças de 0,02 a 0,03 de F1 entre modelos estão dentro da oscilação.

**Para quem terminar:** faça um `GridSearchCV` para o Random Forest com `{'max_depth': [2, 3, 5, None], 'min_samples_leaf': [1, 3, 5]}`. Ele passa a logística com pesos?